# GraphRAG Demo: Two-Phase Query Approach

This notebook demonstrates the GraphRAG implementation using **Azure AI Search** for semantic search and **Cosmos DB Gremlin** for graph exploration.

## Architecture Overview

```
User Query
    │
    ▼
┌─────────────────────────────────────┐
│  PHASE 1: Semantic Search           │
│  (Azure AI Search)                   │
│  - Generate query embedding          │
│  - Hybrid search (text + vector)     │
│  - Return ranked chunks              │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  PHASE 2: Graph Exploration          │
│  (Cosmos DB Gremlin)                 │
│  - Find related chunks               │
│  - Get parent documents/sections     │
│  - Extract keywords                  │
└─────────────────────────────────────┘
    │
    ▼
Enriched Results
```

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure Subscription** with the following resources deployed:
   - Cosmos DB with Gremlin API
   - Azure AI Search
   - Azure OpenAI with text-embedding-3-large

2. **Environment Variables** set:
   - `AZURE_SUBSCRIPTION_ID`
   - `RESOURCE_GROUP_NAME` (default: `rg-graphrag-lab`)
   - `COSMOS_ACCOUNT_NAME` (default: `cosmos-graphrag-dev`)
   - `SEARCH_SERVICE_NAME` (default: `search-graphrag-dev`)
   - `OPENAI_ACCOUNT_NAME` (default: `oai-graphrag-dev`)

3. **Graph initialized** with sample data (`python scripts/initialize_graph.py`)

4. **Search index created** and populated (`python scripts/setup_search_index.py` and `python scripts/setup_search_indexer.py`)

## Step 1: Install Dependencies

In [ ]:
# Install required packages (run once)
# !pip install gremlinpython azure-search-documents azure-identity azure-mgmt-search azure-mgmt-cosmosdb azure-mgmt-cognitiveservices openai python-dotenv

## Step 2: Setup Environment

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add src directory to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Load environment variables from .env file
load_dotenv()

# Verify subscription ID is set
subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
if not subscription_id:
    print("⚠️ AZURE_SUBSCRIPTION_ID not set!")
    print("Please set it: os.environ['AZURE_SUBSCRIPTION_ID'] = 'your-subscription-id'")
else:
    print(f"✅ Subscription ID: {subscription_id[:8]}...")
    print(f"✅ Resource Group: {os.getenv('RESOURCE_GROUP_NAME', 'rg-graphrag-lab')}")
    print(f"✅ Cosmos Account: {os.getenv('COSMOS_ACCOUNT_NAME', 'cosmos-graphrag-dev')}")
    print(f"✅ Search Service: {os.getenv('SEARCH_SERVICE_NAME', 'search-graphrag-dev')}")

## Step 3: Initialize Clients

We need to initialize three clients:
1. **GraphOperations** - For Cosmos DB Gremlin queries
2. **SearchOperations** - For Azure AI Search
3. **EmbeddingsClient** - For generating vector embeddings

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.cosmosdb import CosmosDBManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

# Configuration
subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
resource_group = os.getenv('RESOURCE_GROUP_NAME', 'rg-graphrag-lab')
cosmos_account = os.getenv('COSMOS_ACCOUNT_NAME', 'cosmos-graphrag-dev')
cosmos_database = os.getenv('COSMOS_DATABASE_NAME', 'graphrag-db')
cosmos_graph = os.getenv('COSMOS_GRAPH_NAME', 'knowledge-graph')
search_service = os.getenv('SEARCH_SERVICE_NAME', 'search-graphrag-dev')
search_index = os.getenv('SEARCH_INDEX_NAME', 'chunks-index')
openai_account = os.getenv('OPENAI_ACCOUNT_NAME', 'oai-graphrag-dev')
embedding_deployment = os.getenv('EMBEDDING_DEPLOYMENT_NAME', 'text-embedding-3-large')

print("🔐 Authenticating with Azure...")
credential = DefaultAzureCredential()

# Get Cosmos DB config
print("📊 Getting Cosmos DB configuration...")
cosmos_client = CosmosDBManagementClient(credential, subscription_id)
cosmos_keys = cosmos_client.database_accounts.list_keys(resource_group, cosmos_account)
gremlin_endpoint = f"wss://{cosmos_account}.gremlin.cosmos.azure.com:443/"

# Get Search config
print("🔍 Getting Azure AI Search configuration...")
search_mgmt = SearchManagementClient(credential, subscription_id)
search_keys = search_mgmt.admin_keys.get(resource_group, search_service)
search_endpoint = f"https://{search_service}.search.windows.net"

# Get OpenAI config
print("🤖 Getting Azure OpenAI configuration...")
cognitive_client = CognitiveServicesManagementClient(credential, subscription_id)
openai_account_info = cognitive_client.accounts.get(resource_group, openai_account)
openai_keys = cognitive_client.accounts.list_keys(resource_group, openai_account)
openai_endpoint = openai_account_info.properties.endpoint

print("\n✅ All configurations retrieved successfully!")

In [ ]:
# Import and initialize the operation classes
from graph_operations import GraphOperations
from search_operations import SearchOperations
from embeddings import EmbeddingsClient

print("🔧 Initializing clients...")

# Initialize Graph Operations
graph_ops = GraphOperations(
    endpoint=gremlin_endpoint,
    database=cosmos_database,
    graph=cosmos_graph,
    key=cosmos_keys.primary_master_key
)
print("  ✅ Graph Operations initialized")

# Initialize Search Operations
search_ops = SearchOperations(
    endpoint=search_endpoint,
    index_name=search_index,
    api_key=search_keys.primary_key
)
print("  ✅ Search Operations initialized")

# Initialize Embeddings Client
embeddings_client = EmbeddingsClient(
    endpoint=openai_endpoint,
    api_key=openai_keys.key1,
    deployment_name=embedding_deployment
)
print("  ✅ Embeddings Client initialized")

print("\n🎉 All clients ready!")

## Step 4: Initialize Two-Phase GraphRAG Client

In [ ]:
from custom_query_graphrag import TwoPhaseGraphRAG

# Create the two-phase GraphRAG client
graphrag = TwoPhaseGraphRAG(
    graph_ops=graph_ops,
    search_ops=search_ops,
    embeddings_client=embeddings_client
)

print("✅ TwoPhaseGraphRAG client initialized!")

---

# 🔍 Query Examples

Now let's run some queries to see the two-phase approach in action!

## Query 1: Azure AI Services

In [ ]:
# Run a query about Azure AI Services
results = graphrag.query("azure ai services", top=5)

In [ ]:
# Examine Phase 1 results (Semantic Search)
print("=" * 60)
print("PHASE 1 RESULTS: Semantic Search")
print("=" * 60)

phase1 = results['phase1_search']
print(f"\nQuery: {phase1['query']}")
print(f"Total chunks found: {phase1['total_results']}")

for chunk in phase1['chunks']:
    print(f"\n  [{chunk['rank']}] {chunk['chunkId']}")
    print(f"      Document: {chunk['documentId']}")
    print(f"      Score: {chunk['score']:.4f}")
    print(f"      Text: {chunk['text'][:80]}..." if len(chunk['text']) > 80 else f"      Text: {chunk['text']}")

In [ ]:
# Examine Phase 2 results (Graph Exploration)
print("=" * 60)
print("PHASE 2 RESULTS: Graph Exploration")
print("=" * 60)

phase2 = results['phase2_graph']
print(f"\nChunks explored: {phase2['explored_chunks']}")
print(f"Related chunks discovered: {len(phase2['all_related_chunks'])}")
print(f"Unique documents: {len(phase2['all_documents'])}")
print(f"Unique keywords: {len(phase2['all_keywords'])}")

if phase2['all_keywords']:
    print(f"\nKeywords: {', '.join(sorted(phase2['all_keywords']))}")

if phase2['all_related_chunks']:
    print(f"\nRelated chunks: {', '.join(phase2['all_related_chunks'])}")

## Query 2: Graph Database Concepts

In [ ]:
# Run a query about graph databases
results_graph = graphrag.query("graph database vertices edges", top=3)

In [ ]:
# Quick summary of results
phase1 = results_graph['phase1_search']
phase2 = results_graph['phase2_graph']

print(f"Query: '{phase1['query']}'")
print(f"\n📊 Phase 1: Found {phase1['total_results']} chunks via semantic search")
print(f"🔗 Phase 2: Discovered {len(phase2['all_related_chunks'])} related chunks, {len(phase2['all_keywords'])} keywords")

## Query 3: Search and RAG

In [ ]:
# Run a query about search and RAG
results_rag = graphrag.query("vector search hybrid retrieval RAG", top=5)

In [ ]:
# Display detailed graph context for first chunk
if results_rag['phase2_graph']['graph_data']:
    first_chunk = results_rag['phase2_graph']['graph_data'][0]
    print(f"Detailed Graph Context for: {first_chunk['chunkId']}")
    print("=" * 50)
    print(f"Related Chunks: {first_chunk['relatedChunks']}")
    print(f"Document: {first_chunk['document']}")
    print(f"Section: {first_chunk['section']}")
    print(f"Keywords: {first_chunk['keywords']}")

---

# 🧪 Direct Graph Queries

You can also run direct Gremlin queries to explore the graph structure.

In [ ]:
# Count all vertices by label
print("Vertex counts by label:")
print("=" * 30)

for label in ['document', 'section', 'chunk', 'keyword']:
    result = graph_ops.execute_query(f"g.V().hasLabel('{label}').count()")
    count = result[0] if result else 0
    # Handle nested list
    if isinstance(count, list):
        count = count[0] if count else 0
    print(f"  {label}: {count}")

In [ ]:
# Count all edges by label
print("Edge counts by label:")
print("=" * 30)

for label in ['hasSection', 'hasChunk', 'hasKeyword', 'relatedTo']:
    result = graph_ops.execute_query(f"g.E().hasLabel('{label}').count()")
    count = result[0] if result else 0
    if isinstance(count, list):
        count = count[0] if count else 0
    print(f"  {label}: {count}")

In [ ]:
# Get all keywords in the graph
print("All keywords in the graph:")
print("=" * 30)

keywords = graph_ops.execute_query("g.V().hasLabel('keyword').values('term')")
print(f"  {', '.join(str(k) for k in keywords)}")

In [ ]:
# Get document titles
print("Documents in the graph:")
print("=" * 30)

docs = graph_ops.execute_query("g.V().hasLabel('document').valueMap('docId', 'title')")
for doc in docs:
    if isinstance(doc, dict):
        doc_id = doc.get('docId', ['Unknown'])[0] if isinstance(doc.get('docId'), list) else doc.get('docId', 'Unknown')
        title = doc.get('title', ['Unknown'])[0] if isinstance(doc.get('title'), list) else doc.get('title', 'Unknown')
        print(f"  {doc_id}: {title}")

---

# 🔎 Direct Search Queries

You can also run direct searches against Azure AI Search.

In [ ]:
# Simple text search
print("Text Search Results for 'Azure OpenAI':")
print("=" * 50)

text_results = search_ops.search_text("Azure OpenAI", top=3)
for i, result in enumerate(text_results, 1):
    print(f"\n[{i}] {result['chunkId']}")
    print(f"    Score: {result['score']:.4f}")
    print(f"    Text: {result['text'][:100]}..." if len(result['text']) > 100 else f"    Text: {result['text']}")

In [ ]:
# Vector search
print("Vector Search Results for 'machine learning AI':")
print("=" * 50)

# Generate embedding for the query
query_embedding = embeddings_client.generate_embedding("machine learning AI")
print(f"Query embedding dimensions: {len(query_embedding)}")

vector_results = search_ops.search_vector(query_embedding, top=3)
for i, result in enumerate(vector_results, 1):
    print(f"\n[{i}] {result['chunkId']}")
    print(f"    Score: {result['score']:.4f}")
    print(f"    Text: {result['text'][:100]}..." if len(result['text']) > 100 else f"    Text: {result['text']}")

---

# 🧹 Cleanup

Close the connections when done.

In [ ]:
# Close connections
graphrag.close()
print("✅ All connections closed!")

---

# 📝 Summary

In this notebook, we demonstrated:

1. **Two-Phase Query Approach**:
   - **Phase 1**: Semantic search using Azure AI Search with vector embeddings
   - **Phase 2**: Graph exploration using Cosmos DB Gremlin to find relationships

2. **Key Benefits of GraphRAG**:
   - Combines semantic similarity with explicit relationships
   - Discovers related content through graph traversal
   - Provides richer context with document/section hierarchy
   - Extracts keywords for additional context

3. **Direct Operations**:
   - Running Gremlin queries for graph exploration
   - Using Azure AI Search for text and vector search

## Next Steps

- Add more documents to the graph using `scripts/initialize_graph.py`
- Explore the graph structure with Gremlin queries in `docs/graph_queries.md`
- Integrate with Azure OpenAI for answer generation
- Build a chat interface using the GraphRAG client